# markov analysis

In [ ]:
import pandas as pd

df_analysis = pd.read_csv("../../data/markovian_analysis_baseline.csv")
df_analysis.shape

In [ ]:


state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]
state_num_map = {s: i for i, s in enumerate(state_order)}

df = df_analysis.copy()
df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

df["state"] = df["comfort3_option1"]
df["state_num"] = df["state"].map(state_num_map)

g = df.groupby(["ID", "walk_id"], sort=False)
df["next_state"] = g["state"].shift(-1)
df["next_state_num"] = g["state_num"].shift(-1)
df["next_stop_idx"] = g["stop_idx"].shift(-1)

transitions = df[
    (df["next_stop_idx"] - df["stop_idx"] == 1) &
    (df["stop_idx"] <= 5) &
    (df["next_stop_idx"] <= 5)
].copy()

transitions["delta"] = transitions["next_state_num"] - transitions["state_num"]

transitions[["ID", "walk_id", "stop_idx", "state", "next_state", "delta"]].head()
transitions.shape

# Probem Markov O(1)

In [ ]:
import numpy as np
import pandas as pd

eps = 1e-12
df_analysis = df[df["stop_idx"] <= 5].copy()

# --------------------------------------------------
# 1) Transition counts and transition matrix
# --------------------------------------------------
cm_counts = pd.crosstab(
    transitions["state"],
    transitions["next_state"]
).reindex(index=state_order, columns=state_order, fill_value=0)

P = cm_counts.div(
    cm_counts.sum(axis=1).replace(0, np.nan),
    axis=0
).fillna(0)

# --------------------------------------------------
# 2) Empirical distributions
# --------------------------------------------------

# A) source-state distribution (states that have an outgoing transition)
pi_source = (
    transitions["state"]
    .value_counts(normalize=True)
    .reindex(state_order, fill_value=0)
)

# B) full observed occupancy distribution
pi_occ = (
    df_analysis["comfort3_option1"]
    .value_counts(normalize=True)
    .reindex(state_order, fill_value=0)
)

# --------------------------------------------------
# 3) Stationary distribution implied by P
# Solve pi P = pi, sum(pi)=1
# --------------------------------------------------
A = P.T.values - np.eye(len(state_order))
A[-1, :] = 1.0
b = np.zeros(len(state_order))
b[-1] = 1.0

pi_stat = np.linalg.solve(A, b)
pi_stat = pd.Series(pi_stat, index=state_order, name="pi_stationary")

# numerical cleanup
pi_stat = pi_stat.clip(lower=0)
pi_stat = pi_stat / pi_stat.sum()

# --------------------------------------------------
# 4) Potential-like landscapes
# --------------------------------------------------
landscape = pd.DataFrame({
    "state": state_order,
    "pi_source": pi_source.values,
    "pi_occ": pi_occ.values,
    "pi_stat": pi_stat.reindex(state_order).values,
})

landscape["U_source"] = -np.log(np.clip(landscape["pi_source"], eps, None))
landscape["U_occ"] = -np.log(np.clip(landscape["pi_occ"], eps, None))
landscape["U_stat"] = -np.log(np.clip(landscape["pi_stat"], eps, None))

# optional summaries
valley_state_stat = landscape.loc[landscape["U_stat"].idxmin(), "state"]
valley_depth_stat = landscape["U_stat"].max() - landscape["U_stat"].min()

print("Stationary valley state:", valley_state_stat)
print("Stationary valley depth:", round(float(valley_depth_stat), 4))

landscape.to_csv("markov_results/landscape_comparison.csv", index=False)
landscape

# Per a categories com Gènere/Edat i esos el balanç detallat i estas vainas

In [ ]:
def analyze_markov_by_group(
    df_base,
    transitions_base,
    group_col,
    state_order,
    global_pi_stat,
    group_order=None,
    min_votes=30,
    min_transitions=20,
):
    """
    For each level of group_col:
    - empirical occupancy
    - transition matrix
    - stationary distribution
    - detailed-balance currents
    - comparison to global stationary distribution

    No landscape / potential calculation here.
    """

    d = df_base.dropna(subset=[group_col]).copy()
    tr = transitions_base.dropna(subset=[group_col]).copy()

    if group_order is None:
        levels = sorted(d[group_col].dropna().unique())
    else:
        levels = [x for x in group_order if x in set(d[group_col].dropna().unique())]

    all_distributions = []
    all_summaries = []
    all_currents = []

    transition_matrices = {}
    transition_counts = {}
    current_matrices = {}

    for level in levels:
        d_g = d[d[group_col] == level].copy()
        tr_g = tr[tr[group_col] == level].copy()

        n_votes = len(d_g)
        n_transitions = len(tr_g)

        cm_counts, P = transition_matrix_from_transitions(tr_g, state_order)

        pi_occ = (
            d_g["state"]
            .value_counts(normalize=True)
            .reindex(state_order, fill_value=0)
        )

        pi_source = (
            tr_g["state"]
            .value_counts(normalize=True)
            .reindex(state_order, fill_value=0)
        )

        pi_stat = stationary_distribution(P)

        F, J, pairwise_currents, current_summary = detailed_balance_currents(P, pi_stat)

        dist_g = pd.DataFrame({
            "group_col": group_col,
            "group": level,
            "state": state_order,
            "pi_source": pi_source.values,
            "pi_occ": pi_occ.values,
            "pi_stat": pi_stat.values,
            "pi_global_stat": global_pi_stat.reindex(state_order).values,
        })

        dist_g["delta_pi_stat_vs_global"] = (
            dist_g["pi_stat"] - dist_g["pi_global_stat"]
        )

        # useful kinetic quantities
        onset = (
            P.loc["comfortable-neutral", "uncomfortable"] +
            P.loc["comfortable-neutral", "very uncomfortable"]
        )

        escalation = P.loc["uncomfortable", "very uncomfortable"]
        vu_trap = P.loc["very uncomfortable", "very uncomfortable"]

        recovery = (
            P.loc["uncomfortable", "comfortable-neutral"] +
            P.loc["very uncomfortable", "comfortable-neutral"]
        )

        dist_stat = distribution_distance(pi_stat, global_pi_stat)
        dist_occ = distribution_distance(pi_occ, global_pi_stat)

        # row counts / origin-state transition counts
        origin_counts = cm_counts.sum(axis=1).reindex(state_order, fill_value=0)

        summary_g = {
            "group_col": group_col,
            "group": level,

            "n_votes": n_votes,
            "n_transitions": n_transitions,
            "low_votes_flag": n_votes < min_votes,
            "low_transitions_flag": n_transitions < min_transitions,

            "n_origin_CN": origin_counts["comfortable-neutral"],
            "n_origin_U": origin_counts["uncomfortable"],
            "n_origin_VU": origin_counts["very uncomfortable"],

            "pi_stat_CN": pi_stat["comfortable-neutral"],
            "pi_stat_U": pi_stat["uncomfortable"],
            "pi_stat_VU": pi_stat["very uncomfortable"],

            "pi_occ_CN": pi_occ["comfortable-neutral"],
            "pi_occ_U": pi_occ["uncomfortable"],
            "pi_occ_VU": pi_occ["very uncomfortable"],

            "global_pi_stat_CN": global_pi_stat["comfortable-neutral"],
            "global_pi_stat_U": global_pi_stat["uncomfortable"],
            "global_pi_stat_VU": global_pi_stat["very uncomfortable"],

            "p_onset": onset,
            "p_escalation": escalation,
            "p_vu_trap": vu_trap,
            "p_recovery": recovery,

            "stat_L1_to_global_stat": dist_stat["L1_to_global_stat"],
            "stat_TV_to_global_stat": dist_stat["TV_to_global_stat"],
            "stat_L2_to_global_stat": dist_stat["L2_to_global_stat"],
            "stat_delta_CN": dist_stat["delta_CN_vs_global_stat"],
            "stat_delta_U": dist_stat["delta_U_vs_global_stat"],
            "stat_delta_VU": dist_stat["delta_VU_vs_global_stat"],

            "occ_L1_to_global_stat": dist_occ["L1_to_global_stat"],
            "occ_TV_to_global_stat": dist_occ["TV_to_global_stat"],
            "occ_L2_to_global_stat": dist_occ["L2_to_global_stat"],
            "occ_delta_CN": dist_occ["delta_CN_vs_global_stat"],
            "occ_delta_U": dist_occ["delta_U_vs_global_stat"],
            "occ_delta_VU": dist_occ["delta_VU_vs_global_stat"],

            "total_abs_current": current_summary["total_abs_current"],
            "mean_abs_current": current_summary["mean_abs_current"],
            "max_abs_current": current_summary["max_abs_current"],
            "mean_relative_current": current_summary["mean_relative_current"],
        }

        pairwise_currents = pairwise_currents.copy()
        pairwise_currents["group_col"] = group_col
        pairwise_currents["group"] = level
        pairwise_currents["n_transitions"] = n_transitions

        transition_counts[level] = cm_counts
        transition_matrices[level] = P
        current_matrices[level] = J

        all_distributions.append(dist_g)
        all_summaries.append(summary_g)
        all_currents.append(pairwise_currents)

    distributions = (
        pd.concat(all_distributions, ignore_index=True)
        if all_distributions else pd.DataFrame()
    )

    summary = pd.DataFrame(all_summaries)

    currents = (
        pd.concat(all_currents, ignore_index=True)
        if all_currents else pd.DataFrame()
    )

    return {
        "summary": summary,
        "distributions": distributions,
        "currents": currents,
        "transition_counts": transition_counts,
        "transition_matrices": transition_matrices,
        "current_matrices": current_matrices,
        "levels": levels,
    }

In [ ]:
import matplotlib.pyplot as plt
def plot_pi_stat_and_occ_vs_global(result, group_col, state_order, global_pi_stat):
    distributions = result["distributions"]
    summary = result["summary"].copy()
    levels = result["levels"]

    if distributions.empty:
        print(f"No distributions for {group_col}")
        return

    n_levels = len(levels)

    fig, axes = plt.subplots(
        2,
        n_levels,
        figsize=(4.8 * n_levels, 7.8),
        sharey=True,
        constrained_layout=True,
    )

    if n_levels == 1:
        axes = np.array([[axes[0]], [axes[1]]])

    x = np.arange(len(state_order))
    width = 0.38

    n_trans_map = dict(zip(summary["group"], summary["n_transitions"]))
    n_votes_map = dict(zip(summary["group"], summary["n_votes"]))

    for j, level in enumerate(levels):
        sub = (
            distributions[distributions["group"] == level]
            .set_index("state")
            .reindex(state_order)
        )

        vals_stat = sub["pi_stat"]
        vals_occ = sub["pi_occ"]
        vals_global = global_pi_stat.reindex(state_order)

        n_tr = n_trans_map.get(level, np.nan)
        n_votes = n_votes_map.get(level, np.nan)

        axes[0, j].bar(
            x - width / 2,
            vals_stat.values,
            width=width,
            label=f"{level} stationary"
        )
        axes[0, j].bar(
            x + width / 2,
            vals_global.values,
            width=width,
            label="global stationary",
            alpha=0.6
        )

        axes[0, j].set_title(f"{level}: stationary vs global\nn_trans={n_tr}")
        axes[0, j].set_xticks(x)
        axes[0, j].set_xticklabels([short_labels[s] for s in state_order])
        axes[0, j].set_ylim(0, 1)
        axes[0, j].grid(True, axis="y", alpha=0.3)

        axes[1, j].bar(
            x - width / 2,
            vals_occ.values,
            width=width,
            label=f"{level} occupancy"
        )
        axes[1, j].bar(
            x + width / 2,
            vals_global.values,
            width=width,
            label="global stationary",
            alpha=0.6
        )

        axes[1, j].set_title(f"{level}: occupancy vs global\nn_votes={n_votes}")
        axes[1, j].set_xticks(x)
        axes[1, j].set_xticklabels([short_labels[s] for s in state_order])
        axes[1, j].set_ylim(0, 1)
        axes[1, j].grid(True, axis="y", alpha=0.3)

        if j == 0:
            axes[0, j].set_ylabel("Probability")
            axes[1, j].set_ylabel("Probability")

    axes[0, -1].legend(fontsize=8)
    axes[1, -1].legend(fontsize=8)

    fig.suptitle(
        f"Subgroup distributions compared to global stationary distribution — {group_col}",
        fontsize=15
    )
    plt.show()

In [ ]:
def plot_transition_matrices_reds(result, group_col, state_order):
    levels = result["levels"]
    tmats = result["transition_matrices"]
    counts = result["transition_counts"]
    summary = result["summary"].copy()

    n_levels = len(levels)

    fig, axes = plt.subplots(
        1,
        n_levels,
        figsize=(4.8 * n_levels, 4.7),
        constrained_layout=True,
    )

    if n_levels == 1:
        axes = [axes]

    n_trans_map = dict(zip(summary["group"], summary["n_transitions"]))

    for ax, level in zip(axes, levels):
        P = tmats[level].copy()
        C = counts[level].copy()

        n_tr = n_trans_map.get(level, np.nan)

        origin_counts = C.sum(axis=1).reindex(state_order, fill_value=0)

        y_labels = [
            f"{short_labels.get(s, s)}\n(n={int(origin_counts.loc[s])})"
            for s in state_order
        ]

        im = ax.imshow(
            P.values,
            vmin=0,
            vmax=1,
            aspect="auto",
            cmap="Reds"
        )

        ax.set_xticks(range(len(state_order)))
        ax.set_yticks(range(len(state_order)))

        ax.set_xticklabels([short_labels.get(s, s) for s in state_order])
        ax.set_yticklabels(y_labels)

        ax.set_title(f"{level}\nN transitions={n_tr}")
        ax.set_xlabel("Next state")
        ax.set_ylabel("Current state")

        for i in range(P.shape[0]):
            for j in range(P.shape[1]):
                val = P.iloc[i, j]
                count = C.iloc[i, j]

                text_color = "white" if val > 0.55 else "black"

                ax.text(
                    j,
                    i,
                    f"{val:.2f}\n({int(count)})",
                    ha="center",
                    va="center",
                    fontsize=8,
                    color=text_color,
                )

    cbar = fig.colorbar(im, ax=axes, shrink=0.85)
    cbar.set_label("Transition probability")

    fig.suptitle(
        f"Transition matrices by {group_col}",
        fontsize=15
    )
    plt.show()

In [ ]:
def plot_kinetic_probabilities(result, group_col):
    summary = result["summary"].copy()
    levels = result["levels"]

    if summary.empty:
        print(f"No summary for {group_col}")
        return

    summary["group"] = pd.Categorical(
        summary["group"],
        categories=levels,
        ordered=True
    )
    summary = summary.sort_values("group")

    kinetic_cols = ["p_onset", "p_escalation", "p_vu_trap", "p_recovery"]

    fig, ax = plt.subplots(figsize=(9.5, 5.2), constrained_layout=True)

    x = np.arange(len(summary))
    width = 0.18

    for k, col in enumerate(kinetic_cols):
        ax.bar(
            x + (k - 1.5) * width,
            summary[col],
            width=width,
            label=col
        )

    labels = [
        f"{g}\n(n={int(n)})"
        for g, n in zip(summary["group"].astype(str), summary["n_transitions"])
    ]

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylabel("Probability")
    ax.set_title(f"Kinetic probabilities by {group_col}")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    plt.show()

In [ ]:
def plot_distance_to_global(result, group_col):
    summary = result["summary"].copy()
    levels = result["levels"]

    if summary.empty:
        print(f"No summary for {group_col}")
        return

    summary["group"] = pd.Categorical(
        summary["group"],
        categories=levels,
        ordered=True
    )
    summary = summary.sort_values("group")

    fig, ax = plt.subplots(figsize=(8.5, 4.8), constrained_layout=True)

    x = np.arange(len(summary))
    width = 0.35

    ax.bar(
        x - width / 2,
        summary["stat_TV_to_global_stat"],
        width=width,
        label="stationary vs global stationary"
    )

    ax.bar(
        x + width / 2,
        summary["occ_TV_to_global_stat"],
        width=width,
        label="occupancy vs global stationary"
    )

    labels = [
        f"{g}\n(n={int(n)})"
        for g, n in zip(summary["group"].astype(str), summary["n_transitions"])
    ]

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylabel("Total variation distance")
    ax.set_title(f"Distance to global stationary distribution — {group_col}")
    ax.legend()
    ax.grid(True, axis="y", alpha=0.3)

    plt.show()

In [ ]:
all_results = {}

for group_col in group_vars:
    print("\n" + "=" * 90)
    print(f"Analyzing {group_col}")
    print("=" * 90)

    if group_col not in df_base.columns:
        print(f"Skipping {group_col}: not found.")
        continue

    group_order = group_orders.get(group_col, None)

    result = analyze_markov_by_group(
        df_base=df_base,
        transitions_base=transitions_base,
        group_col=group_col,
        state_order=state_order,
        global_pi_stat=global_pi_stat,
        group_order=group_order,
        min_votes=30,
        min_transitions=20,
    )

    all_results[group_col] = result

    print("\nSUMMARY")
    display(result["summary"].round(4))

    print("\nDISTRIBUTIONS")
    display(result["distributions"].round(4))

    print("\nPAIRWISE DETAILED-BALANCE CURRENTS")
    display(result["currents"].round(5))

    plot_pi_stat_and_occ_vs_global(result, group_col, state_order, global_pi_stat)
    plot_transition_matrices_reds(result, group_col, state_order)
    plot_kinetic_probabilities(result, group_col)
    plot_distance_to_global(result, group_col)

summary_all = pd.concat(
    [r["summary"] for r in all_results.values()],
    ignore_index=True
)

print("\nALL SUBGROUP SUMMARY")
display(summary_all.round(4))

# per 3 rangs d'HDX

In [ ]:
import numpy as np
import pandas as pd

# ============================================================
# CONFIG
# ============================================================

x_col = "<HDX-HDX_fixed+<HDX>>"
regime_labels = ["Low HDX", "Middle HDX", "High HDX"]
eps = 1e-12

# ============================================================
# 1. CLEAN
# ============================================================

df = df.loc[:, ~df.columns.duplicated()].copy()
transitions = transitions.loc[:, ~transitions.columns.duplicated()].copy()

# remove previous regime columns if they exist
for c in ["HDX_regime"]:
    if c in df.columns:
        df = df.drop(columns=c)
    if c in transitions.columns:
        transitions = transitions.drop(columns=c)

df[x_col] = pd.to_numeric(df[x_col], errors="coerce")

# ============================================================
# 2. CREATE 3 HDX REGIMES WITH APPROX SAME NUMBER OF VOTES
# ============================================================

df_regime_base = df[
    (df["stop_idx"] <= 5) &
    (df["state"].isin(state_order)) &
    (df[x_col].notna())
].copy()

q = pd.qcut(
    df_regime_base[x_col],
    q=3,
    duplicates="drop"
)

labels_used = regime_labels[:len(q.cat.categories)]

df_regime_base["HDX_regime"] = (
    q.cat.rename_categories(labels_used)
    .astype(str)
)

print("Votes per HDX regime:")
print(df_regime_base["HDX_regime"].value_counts().reindex(labels_used))

print("\nHDX regime ranges:")
print(
    df_regime_base
    .groupby("HDX_regime")[x_col]
    .agg(
        n="count",
        min="min",
        median="median",
        max="max"
    )
    .reindex(labels_used)
    .round(3)
)

# ============================================================
# 3. ADD HDX REGIME TO TRANSITIONS USING MERGE, NOT INDEX
# ============================================================

# use row_id if available, otherwise fall back to ID + walk_id + stop_idx
if "row_id" in df.columns and "row_id" in transitions.columns:
    merge_keys = ["ID", "walk_id", "stop_idx", "row_id"]
else:
    merge_keys = ["ID", "walk_id", "stop_idx"]

lookup = df_regime_base[merge_keys + ["HDX_regime"]].copy()

transitions_reg = transitions.merge(
    lookup,
    on=merge_keys,
    how="left"
)

transitions_reg = transitions_reg.dropna(subset=["HDX_regime"]).copy()

transitions_reg = transitions_reg[
    transitions_reg["state"].isin(state_order) &
    transitions_reg["next_state"].isin(state_order)
].copy()

print("\nTransitions per HDX regime:")
print(transitions_reg["HDX_regime"].value_counts().reindex(labels_used))

# ============================================================
# 4. STATIONARY DISTRIBUTION
# ============================================================

def stationary_distribution(P):
    """
    Solve pi P = pi, sum(pi)=1.
    """
    n = P.shape[0]
    P_mat = P.values.astype(float)

    A = P_mat.T - np.eye(n)
    A[-1, :] = 1.0

    b = np.zeros(n)
    b[-1] = 1.0

    try:
        pi = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        eigvals, eigvecs = np.linalg.eig(P_mat.T)
        idx = np.argmin(np.abs(eigvals - 1))
        pi = np.real(eigvecs[:, idx])

    pi = np.clip(pi, 0, None)

    if pi.sum() == 0:
        pi = np.ones(n) / n
    else:
        pi = pi / pi.sum()

    return pd.Series(pi, index=P.index)

# ============================================================
# 5. MARKOV ANALYSIS BY HDX REGIME
# ============================================================

all_landscapes = []
all_summaries = []
transition_matrices = {}
transition_counts = {}

for regime in labels_used:

    df_occ_r = df_regime_base[
        df_regime_base["HDX_regime"] == regime
    ].copy()

    trans_r = transitions_reg[
        transitions_reg["HDX_regime"] == regime
    ].copy()

    # transition counts
    cm_counts = pd.crosstab(
        trans_r["state"],
        trans_r["next_state"]
    ).reindex(
        index=state_order,
        columns=state_order,
        fill_value=0
    )

    # transition matrix
    row_sums = cm_counts.sum(axis=1)

    P = cm_counts.div(
        row_sums.replace(0, np.nan),
        axis=0
    )

    # if a state has no outgoing transitions in that regime, add self-loop
    for s in state_order:
        if row_sums.loc[s] == 0:
            P.loc[s, :] = 0.0
            P.loc[s, s] = 1.0

    P = P.fillna(0)

    # source-state distribution
    pi_source = (
        trans_r["state"]
        .value_counts(normalize=True)
        .reindex(state_order, fill_value=0)
    )

    # occupancy distribution
    pi_occ = (
        df_occ_r["state"]
        .value_counts(normalize=True)
        .reindex(state_order, fill_value=0)
    )

    # stationary distribution
    pi_stat = stationary_distribution(P)

    landscape_r = pd.DataFrame({
        "regime": regime,
        "state": state_order,
        "pi_source": pi_source.values,
        "pi_occ": pi_occ.values,
        "pi_stat": pi_stat.values,
    })

    landscape_r["U_source"] = -np.log(np.clip(landscape_r["pi_source"], eps, None))
    landscape_r["U_occ"] = -np.log(np.clip(landscape_r["pi_occ"], eps, None))
    landscape_r["U_stat"] = -np.log(np.clip(landscape_r["pi_stat"], eps, None))

    valley_state = landscape_r.loc[
        landscape_r["U_stat"].idxmin(),
        "state"
    ]

    valley_depth = landscape_r["U_stat"].max() - landscape_r["U_stat"].min()

    summary_r = {
        "regime": regime,
        "n_votes": len(df_occ_r),
        "n_transitions": len(trans_r),
        "valley_state": valley_state,
        "valley_depth": valley_depth,
        "pi_stat_comfortable_neutral": pi_stat["comfortable-neutral"],
        "pi_stat_uncomfortable": pi_stat["uncomfortable"],
        "pi_stat_very_uncomfortable": pi_stat["very uncomfortable"],
        "pi_occ_comfortable_neutral": pi_occ["comfortable-neutral"],
        "pi_occ_uncomfortable": pi_occ["uncomfortable"],
        "pi_occ_very_uncomfortable": pi_occ["very uncomfortable"],
    }

    transition_counts[regime] = cm_counts
    transition_matrices[regime] = P

    all_landscapes.append(landscape_r)
    all_summaries.append(summary_r)

landscape_by_regime = pd.concat(all_landscapes, ignore_index=True)
summary_by_regime = pd.DataFrame(all_summaries)

print("\nSummary by HDX regime:")
print(summary_by_regime.round(4))

print("\nStationary distributions:")
print(
    landscape_by_regime
    .pivot(index="state", columns="regime", values="pi_stat")
    .reindex(state_order)
    .reindex(columns=labels_used)
    .round(3)
)

print("\nTransition matrices:")
for regime in labels_used:
    print("\n" + regime)
    print(transition_matrices[regime].round(3))

In [ ]:
import matplotlib.pyplot as plt

pi_occ_plot = (
    landscape_by_regime
    .pivot(index="state", columns="regime", values="pi_occ")
    .reindex(state_order)
    .reindex(columns=labels_used)
)

fig, axes = plt.subplots(
    nrows=1,
    ncols=len(labels_used),
    figsize=(5 * len(labels_used), 4.5),
    sharey=True,
    constrained_layout=True,
)

if len(labels_used) == 1:
    axes = [axes]

for ax, regime in zip(axes, labels_used):
    vals = pi_occ_plot[regime]

    ax.bar(
        vals.index,
        vals.values,
    )

    ax.set_title(regime)
    ax.set_xlabel("Comfort state")
    ax.set_ylabel("Occupancy probability")
    ax.grid(True, axis="y", alpha=0.3)
    ax.set_ylim(0, max(0.05, pi_occ_plot.max().max() * 1.15))
    ax.tick_params(axis="x", rotation=25)

fig.suptitle(
    "Empirical occupancy distributions by HDX regime",
    fontsize=15,
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import gridspec

# ============================================================
# STATIONARY DISTRIBUTIONS BY HDX REGIME
# ============================================================

pi_stat_plot = (
    landscape_by_regime
    .pivot(index="state", columns="regime", values="pi_stat")
    .reindex(state_order)
    .reindex(columns=labels_used)
)

# ============================================================
# OVERALL / NORMAL EMPIRICAL OCCUPANCY DISTRIBUTION
# ============================================================

overall_occ = (
    df[
        (df["stop_idx"] <= 5) &
        (df["state"].isin(state_order))
    ]["state"]
    .value_counts(normalize=True)
    .reindex(state_order, fill_value=0)
)

# common y-limit for visual comparison
ymax = max(
    pi_stat_plot.max().max(),
    overall_occ.max()
) * 1.15

# ============================================================
# FIGURE LAYOUT
# top: stationary distributions by regime
# bottom: overall empirical occupancy
# ============================================================

fig = plt.figure(figsize=(14, 9))

gs = gridspec.GridSpec(
    2, 3,
    height_ratios=[1, 1.1],
    hspace=0.38,
    wspace=0.30,
)

# ----------------------------
# Top row: stationary distributions
# ----------------------------
for j, regime in enumerate(labels_used):
    ax = fig.add_subplot(gs[0, j])

    vals = pi_stat_plot[regime]

    ax.bar(
        vals.index,
        vals.values,
    )

    ax.set_title(f"{regime} — stationary")
    ax.set_ylabel("Probability")
    ax.set_ylim(0, 0.75)
    ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=25)

# ----------------------------
# Bottom row: overall occupancy
# ----------------------------
ax_bottom = fig.add_subplot(gs[1, :])

ax_bottom.bar(
    overall_occ.index,
    overall_occ.values,
)

ax_bottom.set_title("Overall empirical occupancy distribution")
ax_bottom.set_ylabel("Probability")
ax_bottom.set_xlabel("Comfort state")
ax_bottom.set_ylim(0, ymax)
ax_bottom.grid(True, axis="y", alpha=0.3)
ax_bottom.tick_params(axis="x", rotation=25)

fig.suptitle(
    "Stationary distributions by HDX regime and overall empirical occupancy",
    fontsize=15,
    y=0.98,
)

plt.show()

In [ ]:
pi_vec = pi_stat.reindex(state_order).to_numpy(dtype=float)
P_mat = P.reindex(index=state_order, columns=state_order).to_numpy(dtype=float)

# pair flows F_ij = pi_i P_ij
flow = pi_vec[:, None] * P_mat

# antisymmetric probability currents
J = flow - flow.T

# use only unique pairs
upper_idx = np.triu_indices_from(J, k=1)
abs_J_pairs = np.abs(J[upper_idx])

# relative departure from detailed balance
eps = 1e-12
flow_pair_sum = flow + flow.T
rel_J = np.abs(J) / np.clip(flow_pair_sum, eps, None)
rel_J_pairs = rel_J[upper_idx]

db_summary = pd.DataFrame({
    "metric": [
        "mean_abs_db_departure",
        "max_abs_db_departure",
        "sum_abs_db_departure",
        "mean_rel_db_departure",
        "max_rel_db_departure",
    ],
    "value": [
        float(abs_J_pairs.mean()),
        float(abs_J_pairs.max()),
        float(abs_J_pairs.sum()),
        float(rel_J_pairs.mean()),
        float(rel_J_pairs.max()),
    ],
})

net_currents = []
for i, si in enumerate(state_order):
    for j, sj in enumerate(state_order):
        if i < j:
            fij = float(flow[i, j])
            fji = float(flow[j, i])
            Jij = float(J[i, j])
            net_currents.append({
                "state_i": si,
                "state_j": sj,
                "flow_ij": fij,
                "flow_ji": fji,
                "J_ij": Jij,
                "abs_J": abs(Jij),
                "rel_J": abs(Jij) / max(fij + fji, eps),
                "direction": f"{si}->{sj}" if Jij > 0 else f"{sj}->{si}",
            })

net_currents_df = pd.DataFrame(net_currents).sort_values("abs_J", ascending=False)
db_summary
net_currents_df

# Ara buildejem triplets per a comprar O(1) i O(2)
 Triplets en sentit de prev.state, state i nxt state

In [ ]:
state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]
state_num_map = {s: i for i, s in enumerate(state_order)}

df = df_analysis.copy()
df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

df["state"] = df["comfort3_option1"]
df["state_num"] = df["state"].map(state_num_map)

g = df.groupby(["ID", "walk_id"], sort=False)

df["prev_state"] = g["state"].shift(1)
df["prev_state_num"] = g["state_num"].shift(1)
df["prev_stop_idx"] = g["stop_idx"].shift(1)

df["next_state"] = g["state"].shift(-1)
df["next_state_num"] = g["state_num"].shift(-1)
df["next_stop_idx"] = g["stop_idx"].shift(-1)

triplets = df[
    (df["stop_idx"] <= 5) &
    (df["prev_stop_idx"] >= 1) &
    (df["next_stop_idx"] <= 5) &
    ((df["stop_idx"] - df["prev_stop_idx"]) == 1) &
    ((df["next_stop_idx"] - df["stop_idx"]) == 1)
].copy()

triplets["delta_prev_to_curr"] = triplets["state_num"] - triplets["prev_state_num"]
triplets["delta_curr_to_next"] = triplets["next_state_num"] - triplets["state_num"]

triplets[[
    "ID", "walk_id", "prev_stop_idx", "stop_idx", "next_stop_idx",
    "prev_state", "state", "next_state"
]].head()

triplets.to_csv("markov_results/markov_triplets.csv", index=False)

# O(1)

In [ ]:
cm1_counts = pd.crosstab(
    transitions["state"],
    transitions["next_state"]
).reindex(index=state_order, columns=state_order, fill_value=0)

P1 = cm1_counts.div(cm1_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)
P1

# O(2)

In [ ]:
cm2_counts = (
    triplets.groupby(["prev_state", "state", "next_state"])
    .size()
    .rename("count")
    .reset_index()
)

cm2_counts["prob"] = cm2_counts.groupby(["prev_state", "state"])["count"].transform(
    lambda x: x / x.sum()
)

cm2_counts.head(12)

# comparing O(1) vs O(2)

In [ ]:
rows = []

for prev_state in state_order:
    for state in state_order:
        sub = cm2_counts[
            (cm2_counts["prev_state"] == prev_state) &
            (cm2_counts["state"] == state)
        ].copy()

        if sub.empty:
            continue

        p2 = (
            sub.set_index("next_state")["prob"]
            .reindex(state_order, fill_value=0.0)
        )

        p1 = P1.loc[state].reindex(state_order, fill_value=0.0)

        n = int(sub["count"].sum())
        l1 = float(np.abs(p2 - p1).sum())
        max_abs = float(np.abs(p2 - p1).max())

        rows.append({
            "prev_state": prev_state,
            "state": state,
            "n_triplets": n,
            "L1_diff_order2_vs_order1": l1,
            "max_abs_diff": max_abs,
            "p1_next_comf_neutral": float(p1.iloc[0]),
            "p2_next_comf_neutral": float(p2.iloc[0]),
            "p1_next_uncomfortable": float(p1.iloc[1]),
            "p2_next_uncomfortable": float(p2.iloc[1]),
            "p1_next_very_uncomfortable": float(p1.iloc[2]),
            "p2_next_very_uncomfortable": float(p2.iloc[2]),
        })

markov_compare = pd.DataFrame(rows).sort_values(
    ["L1_diff_order2_vs_order1", "n_triplets"],
    ascending=[False, False]
)

markov_compare.to_csv("markov_results/markov_comparison.csv", index=False)
markov_compare

# versió llarga

In [ ]:
rows_long = []

for prev_state in state_order:
    for state in state_order:
        sub = cm2_counts[
            (cm2_counts["prev_state"] == prev_state) &
            (cm2_counts["state"] == state)
        ].copy()

        if sub.empty:
            continue

        p2 = (
            sub.set_index("next_state")["prob"]
            .reindex(state_order, fill_value=0.0)
        )
        p1 = P1.loc[state].reindex(state_order, fill_value=0.0)

        n = int(sub["count"].sum())
        l1 = float(np.abs(p2 - p1).sum())
        max_abs = float(np.abs(p2 - p1).max())

        for ns in state_order:
            rows_long.append({
                "prev_state": prev_state,
                "state": state,
                "next_state": ns,
                "n_triplets": n,
                "L1_diff_order2_vs_order1": l1,
                "max_abs_diff": max_abs,
                "p1_prob": float(p1[ns]),
                "p2_prob": float(p2[ns]),
                "abs_diff": float(abs(p2[ns] - p1[ns])),
            })

markov_compare_long = pd.DataFrame(rows_long)
markov_compare_long.to_csv("markov_results/markov_comparison_comf_long.csv", index=False)

## Podem actualy veure diferències reals entre O(1) i O(2)

In [ ]:
weighted_l1 = np.average(
    markov_compare["L1_diff_order2_vs_order1"],
    weights=markov_compare["n_triplets"]
)

summary_markov = pd.DataFrame({
    "metric": ["weighted_mean_L1_order2_vs_order1", "n_triplet_contexts"],
    "value": [float(weighted_l1), int(len(markov_compare))]
})

summary_markov.to_csv("markov_results/summary_markov.csv", index=False)

# Anem a veure el raw data a veure el corrents

In [ ]:
import numpy as np
import pandas as pd

# raw transition counts
N = pd.crosstab(
    transitions["state"],
    transitions["next_state"]
).reindex(index=state_order, columns=state_order, fill_value=0)

# empirical pair flows
N_total = N.values.sum()
F_emp = N / N_total

# antisymmetric raw empirical currents
J_emp = F_emp - F_emp.T

# unique pairs only
pairs = []
eps = 1e-12

for i, si in enumerate(state_order):
    for j, sj in enumerate(state_order):
        if i < j:
            fij = float(F_emp.iloc[i, j])
            fji = float(F_emp.iloc[j, i])
            Jij = float(J_emp.iloc[i, j])

            pairs.append({
                "state_i": si,
                "state_j": sj,
                "count_ij": int(N.iloc[i, j]),
                "count_ji": int(N.iloc[j, i]),
                "F_ij_emp": fij,
                "F_ji_emp": fji,
                "J_ij_emp": Jij,
                "abs_J_emp": abs(Jij),
                "rel_J_emp": abs(Jij) / max(fij + fji, eps),
                "direction": f"{si}->{sj}" if Jij > 0 else f"{sj}->{si}",
            })

emp_currents_df = pd.DataFrame(pairs).sort_values("abs_J_emp", ascending=False)

# summary
abs_J_vals = emp_currents_df["abs_J_emp"].to_numpy()
rel_J_vals = emp_currents_df["rel_J_emp"].to_numpy()

emp_db_summary = pd.DataFrame({
    "metric": [
        "mean_abs_emp_current",
        "max_abs_emp_current",
        "sum_abs_emp_current",
        "mean_rel_emp_current",
        "max_rel_emp_current",
    ],
    "value": [
        float(abs_J_vals.mean()),
        float(abs_J_vals.max()),
        float(abs_J_vals.sum()),
        float(rel_J_vals.mean()),
        float(rel_J_vals.max()),
    ],
})

print("Raw count matrix N:")
display(N)

print("Empirical current summary:")
display(emp_db_summary)

print("Pairwise raw empirical currents:")
display(emp_currents_df)

N.to_csv("markov_results/empirical_count_matrix.csv")
emp_db_summary.to_csv("markov_results/empirical_current_summary.csv", index=False)
emp_currents_df.to_csv("markov_results/empirical_currents.csv", index=False)

# Corrents empíriques (MICROSCÒPIC)
### Primer construim la sequencia

In [ ]:
state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]
state_to_num = {s: i for i, s in enumerate(state_order)}
num_to_state = {i: s for s, i in state_to_num.items()}

In [ ]:
def extract_person_trajectories(df, state_col="comfort3_option1", max_stop=7):
    df = df.copy()
    df = df[df["stop_idx"] <= max_stop].copy()
    df = df.sort_values(["ID", "walk_id", "stop_idx", "row_id"]).reset_index(drop=True)

    trajectories = []

    for (ID, walk_id), sub in df.groupby(["ID", "walk_id"], sort=False):
        sub = sub.dropna(subset=[state_col]).copy()
        if len(sub) < 2:
            continue

        stops = sub["stop_idx"].tolist()
        states = sub[state_col].tolist()

        # quedem-nos només runs consecutives
        seq_states = [states[0]]
        seq_stops = [stops[0]]

        for k in range(1, len(states)):
            if stops[k] - stops[k-1] == 1:
                seq_states.append(states[k])
                seq_stops.append(stops[k])
            else:
                if len(seq_states) >= 2:
                    trajectories.append({
                        "ID": ID,
                        "walk_id": walk_id,
                        "stops": seq_stops,
                        "states": seq_states,
                        "state_nums": [state_to_num[s] for s in seq_states],
                    })
                seq_states = [states[k]]
                seq_stops = [stops[k]]

        if len(seq_states) >= 2:
            trajectories.append({
                "ID": ID,
                "walk_id": walk_id,
                "stops": seq_stops,
                "states": seq_states,
                "state_nums": [state_to_num[s] for s in seq_states],
            })

    return trajectories

In [ ]:
traj_list = extract_person_trajectories(
    df_analysis,
    state_col="comfort3_option1",
    max_stop=7
)

len(traj_list), traj_list[:3]

# Ara detectem els patterns de vot

In [ ]:
def contains_subsequence(seq, pattern):
    m = len(pattern)
    if len(seq) < m:
        return False
    for i in range(len(seq) - m + 1):
        if seq[i:i+m] == pattern:
            return True
    return False

def count_subsequence(seq, pattern):
    m = len(pattern)
    count = 0
    if len(seq) < m:
        return 0
    for i in range(len(seq) - m + 1):
        if seq[i:i+m] == pattern:
            count += 1
    return count

In [ ]:
def analyze_micro_patterns(traj_list):
    rows = []

    patterns = {
        "contains_012": [0, 1, 2],
        "contains_210": [2, 1, 0],
        "contains_0120": [0, 1, 2, 0],
        "contains_0210": [0, 2, 1, 0],
        "contains_120": [1, 2, 0],
        "contains_201": [2, 0, 1],
    }

    

    for tr in traj_list:
        seq = tr["state_nums"]

        row = {
            "ID": tr["ID"],
            "walk_id": tr["walk_id"],
            "n_states": len(seq),
            "sequence_num": " -> ".join(map(str, seq)),
            "sequence_label": " -> ".join(tr["states"]),
        }

        for name, patt in patterns.items():
            row[name] = contains_subsequence(seq, patt)
            row[name.replace("contains", "count")] = count_subsequence(seq, patt)

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
micro_df = analyze_micro_patterns(traj_list)
micro_df.head()
micro_df.shape

# Resultats 

In [ ]:
def summarize_micro_patterns(micro_df):
    pattern_cols = [c for c in micro_df.columns if c.startswith("contains_")]

    rows = []
    n_traj = len(micro_df)

    for col in pattern_cols:
        n = int(micro_df[col].sum())
        rows.append({
            "pattern": col,
            "n_trajectories": n,
            "fraction_trajectories": n / n_traj if n_traj > 0 else np.nan
        })

    return pd.DataFrame(rows).sort_values("fraction_trajectories", ascending=False)

micro_summary = summarize_micro_patterns(micro_df)
micro_summary

In [ ]:
forward_backward_summary = pd.DataFrame({
    "pattern": ["0->1->2", "2->1->0", "0->1->2->0", "0->2->1->0"],
    "n_trajectories": [
        int(micro_df["contains_012"].sum()),
        int(micro_df["contains_210"].sum()),
        int(micro_df["contains_0120"].sum()),
        int(micro_df["contains_0210"].sum()),
    ]
})

forward_backward_summary

# PLOTS DE O(2) VS O(1) AMB SIGNED DIFFERENCE

In [ ]:
import numpy as np
import pandas as pd

# state order for comfort3_option1
state_order = ["comfortable-neutral", "uncomfortable", "very uncomfortable"]

# order-1 matrix
cm1_counts = pd.crosstab(
    transitions["state"],
    transitions["next_state"]
).reindex(index=state_order, columns=state_order, fill_value=0)

P1 = cm1_counts.div(cm1_counts.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

# order-2 counts
cm2_counts = (
    triplets.groupby(["prev_state", "state", "next_state"])
    .size()
    .rename("count")
    .reset_index()
)

cm2_counts["prob"] = cm2_counts.groupby(["prev_state", "state"])["count"].transform(
    lambda x: x / x.sum()
)

# build long comparison table
rows_long = []

for prev_state in state_order:
    for state in state_order:
        sub = cm2_counts[
            (cm2_counts["prev_state"] == prev_state) &
            (cm2_counts["state"] == state)
        ].copy()

        if sub.empty:
            continue

        p2 = (
            sub.set_index("next_state")["prob"]
            .reindex(state_order, fill_value=0.0)
        )

        p1 = P1.loc[state].reindex(state_order, fill_value=0.0)

        n = int(sub["count"].sum())
        l1 = float(np.abs(p2 - p1).sum())
        max_abs = float(np.abs(p2 - p1).max())

        for ns in state_order:
            rows_long.append({
                "prev_state": prev_state,
                "state": state,
                "next_state": ns,
                "n_triplets": n,
                "L1_diff_order2_vs_order1": l1,
                "max_abs_diff": max_abs,
                "p1_prob": float(p1[ns]),
                "p2_prob": float(p2[ns]),
                "signed_diff": float(p2[ns] - p1[ns]),
                "abs_diff": float(abs(p2[ns] - p1[ns])),
            })

markov_compare_long_3 = pd.DataFrame(rows_long)
markov_compare_long_3.head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_signed_diff_heatmap(markov_long, state_order, current_state, min_n_to_flag=None):
    sub = markov_long[markov_long["state"] == current_state].copy()

    heat = (
        sub.pivot(index="prev_state", columns="next_state", values="signed_diff")
        .reindex(index=state_order, columns=state_order)
    )

    counts = (
        sub.groupby(["prev_state", "next_state"])["n_triplets"]
        .first()
        .unstack("next_state")
        .reindex(index=state_order, columns=state_order)
    )

    vmax = np.nanmax(np.abs(heat.values))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1e-6

    fig, ax = plt.subplots(figsize=(7, 5))
    im = ax.imshow(heat.values, aspect="auto", vmin=-vmax, vmax=vmax, cmap="coolwarm")

    ax.set_xticks(range(len(state_order)))
    ax.set_xticklabels(state_order, rotation=45, ha="right")
    ax.set_yticks(range(len(state_order)))
    ax.set_yticklabels(state_order)

    for i in range(len(state_order)):
        for j in range(len(state_order)):
            val = heat.iloc[i, j]
            n = counts.iloc[i, j] if pd.notna(counts.iloc[i, j]) else np.nan
            if pd.notna(val):
                txt = f"{val:+.2f}"
                if pd.notna(n):
                    txt += f"\n(n={int(n)})"
                ax.text(j, i, txt, ha="center", va="center", fontsize=9)

                if min_n_to_flag is not None and pd.notna(n) and n < min_n_to_flag:
                    ax.add_patch(plt.Rectangle((j-0.5, i-0.5), 1, 1,
                                               fill=False, edgecolor="black",
                                               linewidth=1.5, linestyle="--"))

    ax.set_xlabel("next state")
    ax.set_ylabel("previous state")
    ax.set_title(f"Signed difference: order 2 - order 1\nCurrent state = {current_state}")

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("p2(next|prev,current) - p1(next|current)")

    plt.tight_layout()
    plt.show()

In [ ]:
for current_state in state_order:
    plot_signed_diff_heatmap(
        markov_compare_long_3,
        state_order=state_order,
        current_state=current_state,
        min_n_to_flag=20  # optional: dashed box if context is sparse
    )

# energy plots